# 04 — Bayesian hyperparameter optimisation (Optuna TPE)

This notebook tunes the strongest baseline family selected from Notebook 03. It uses Optuna's Tree-structured Parzen Estimator (TPE) sampler, a Bayesian optimisation approach. The objective is mean GroupKFold macro-F1 on **training trips only**. Student-test data is not used to choose hyperparameters.

The search record is retained as CSV and SQLite so the optimisation process is auditable.

In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import optuna
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score

TRAIN_CSV = Path("data/processed/train_candidates.csv")
MANIFEST = Path("configs/person_b_feature_manifest.json")
BASELINE = Path("experiments/results/person_b/baseline_model_comparison.csv")
OUT = Path("experiments/results/person_b"); OUT.mkdir(parents=True, exist_ok=True)
if not all(p.exists() for p in [TRAIN_CSV, MANIFEST, BASELINE]):
    raise FileNotFoundError("Complete Notebooks 02 and 03 first.")
try:
    import optuna
except ImportError as exc:
    raise ImportError("Install Optuna with: python -m pip install optuna") from exc

df = pd.read_csv(TRAIN_CSV)
features = json.loads(MANIFEST.read_text())["feature_columns"]
X, y, groups = df[features], df["is_match"].astype(int), df["trip_id"]
base = pd.read_csv(BASELINE, index_col=0)
family = str(base["macro_f1_mean"].idxmax())
print("Best baseline family:", family)
if family not in {"random_forest", "hist_gradient_boosting"}:
    print("Using RandomForest as a tunable tree-ensemble family; baseline winner remains documented above.")
    family = "random_forest"
cv = GroupKFold(n_splits=min(5, groups.nunique()))

# Keep search bounded; adjust n_trials only with enough compute time.
N_TRIALS = 30
SEED = 42

In [ ]:
def objective(trial):
    if family == "random_forest":
        estimator = RandomForestClassifier(
            n_estimators=trial.suggest_int("n_estimators", 150, 500, step=50),
            max_depth=trial.suggest_int("max_depth", 8, 32),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
            max_features=trial.suggest_categorical("max_features", ["sqrt", 0.5, 0.8]),
            class_weight="balanced_subsample", n_jobs=-1, random_state=SEED,
        )
    else:
        estimator = HistGradientBoostingClassifier(
            learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
            max_iter=trial.suggest_int("max_iter", 100, 350, step=25),
            max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 15, 63),
            l2_regularization=trial.suggest_float("l2_regularization", 1e-4, 10, log=True),
            random_state=SEED,
        )
    scores=[]
    for tr, va in cv.split(X, y, groups):
        pipe = make_pipeline(SimpleImputer(strategy="median"), estimator.__class__(**estimator.get_params()))
        pipe.fit(X.iloc[tr], y.iloc[tr])
        pred = pipe.predict(X.iloc[va])
        scores.append(f1_score(y.iloc[va], pred, average="macro", zero_division=0))
    return float(np.mean(scores))

storage = f"sqlite:///{(OUT / 'optuna_study.db').resolve().as_posix()}"
study = optuna.create_study(
    study_name="pathsnap_person_b_tpe", direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    storage=storage, load_if_exists=True,
)
start = time.perf_counter()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
optimisation_seconds = time.perf_counter() - start
trials = study.trials_dataframe()
trials.to_csv(OUT / "bayesian_optimization_trials.csv", index=False)
summary = {
    "baseline_family": family,
    "best_value_macro_f1_groupkfold": study.best_value,
    "best_params": study.best_params,
    "n_trials_total": len(study.trials),
    "optimisation_seconds_this_run": optimisation_seconds,
    "sampler": "Optuna TPESampler",
    "objective": "mean GroupKFold macro-F1 on training trips",
}
(OUT / "bayesian_optimization_best.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

In [ ]:
import matplotlib.pyplot as plt
complete = trials[trials["state"] == "COMPLETE"].copy()
plt.figure(figsize=(9, 4))
plt.plot(np.arange(1, len(complete)+1), complete["value"].cummax(), marker="o", markersize=3)
plt.xlabel("Completed trial")
plt.ylabel("Best-so-far GroupKFold macro-F1")
plt.title("Optuna TPE optimisation history")
plt.grid(alpha=.25); plt.tight_layout(); plt.show()
print("Retained optimisation database and trial CSV in", OUT)

## Important

This notebook's output is valid only after it has actually been run on the team's candidate table. Do not report the placeholder notebook defaults as results. Keep the trial database and CSV with the final submission. If the baseline winner is not one of the two tuned families, expand the tuning notebook to that family and document the decision before finalising.